# AE Case Evaluation

This notebook loads a trained AE checkpoint, reads a small validation/test batch, computes reconstruction metrics, and saves compact visualization artifacts. It is intended to be executed on a Slurm spot GPU node via `scripts/submit_eval_notebook_spot.sh`.

In [ ]:
import os
import sys
import json
import glob
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO_ROOT = Path(os.environ.get("REPO_ROOT", Path.cwd())).resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.config import load_config
from utils.builder import ConfigBuilder
from trainers.ae_kl_trainer import STD_LAYER

def repo_path(path_value):
    if not path_value:
        return ""
    path = Path(path_value)
    return path if path.is_absolute() else REPO_ROOT / path

CFG_PATH = str(repo_path(os.environ.get("CFG_PATH", "configs/ae_kl_552_16_evit_full.yaml")))
CKPT_PATH = str(repo_path(os.environ.get("CKPT_PATH", "")))
SPLIT = os.environ.get("SPLIT", "valid")
MAX_BATCHES = int(os.environ.get("MAX_BATCHES", "2"))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "2"))
DATA_NUM_WORKERS = int(os.environ.get("DATA_NUM_WORKERS", "0"))
EVAL_OUTDIR = repo_path(os.environ.get("EVAL_OUTDIR", "eval_outputs"))
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

EVAL_OUTDIR.mkdir(parents=True, exist_ok=True)
print({
    "cfg": CFG_PATH,
    "checkpoint": CKPT_PATH or "<auto>",
    "split": SPLIT,
    "max_batches": MAX_BATCHES,
    "batch_size": BATCH_SIZE,
    "data_num_workers": DATA_NUM_WORKERS,
    "device": str(DEVICE),
    "outdir": str(EVAL_OUTDIR),
})


In [ ]:
def find_latest_best_checkpoint():
    patterns = ["output*/**/best.pth", "output*/**/final.pth", "output*/**/snapshot_latest.pth"]
    candidates = []
    for pattern in patterns:
        candidates.extend(str(path) for path in REPO_ROOT.glob(pattern))
    candidates = [p for p in candidates if Path(p).is_file()]
    if not candidates:
        return ""
    return max(candidates, key=lambda p: Path(p).stat().st_mtime)

if not CKPT_PATH:
    CKPT_PATH = find_latest_best_checkpoint()
if not CKPT_PATH:
    raise FileNotFoundError("No checkpoint found. Set CKPT_PATH=/path/to/best.pth when submitting the notebook.")

cfg = load_config(CFG_PATH)
cfg.setdefault("trainer", {})
cfg["trainer"][f"{SPLIT}_batch_size"] = BATCH_SIZE
cfg["trainer"]["batch_size"] = min(BATCH_SIZE, int(cfg["trainer"].get("batch_size", BATCH_SIZE)))

builder = ConfigBuilder(cfg)
model = builder.build_model().to(DEVICE)
checkpoint = torch.load(CKPT_PATH, map_location="cpu")
if isinstance(checkpoint, dict) and "MODEL_STATE" in checkpoint:
    state = checkpoint["MODEL_STATE"]
else:
    state = checkpoint
state = {k.removeprefix("module."): v for k, v in state.items()}
missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()
print(f"Loaded checkpoint: {CKPT_PATH}")
print(f"Missing keys: {len(missing)}, unexpected keys: {len(unexpected)}")
if missing:
    print("First missing keys:", missing[:5])
if unexpected:
    print("First unexpected keys:", unexpected[:5])


In [ ]:
loader, dataset, _ = builder.build_dataloader(SPLIT, DATA_NUM_WORKERS, distributed=False)
channel_names = []
for name in getattr(dataset, "single_level_vnames", []):
    channel_names.append(name)
for name in getattr(dataset, "multi_level_vnames", []):
    for height in getattr(dataset, "height_level_list", []):
        channel_names.append(f"{name}{height}")
print(f"Dataset split={SPLIT}, len={len(dataset)}, batches={len(loader)}, channels={len(channel_names)}")
print("First channels:", channel_names[:12])


In [ ]:
def pick_channels(names, max_channels=6):
    preferred = ["msl", "u10", "v10", "t2m", "z500", "q500", "u500", "v500", "t500"]
    picks = []
    for p in preferred:
        if p in names and p not in picks:
            picks.append(names.index(p))
    if len(picks) < max_channels:
        for idx in np.linspace(0, max(len(names) - 1, 0), num=min(max_channels, len(names)), dtype=int):
            if int(idx) not in picks:
                picks.append(int(idx))
            if len(picks) >= max_channels:
                break
    return picks[:max_channels]

selected_channels = pick_channels(channel_names)
print("Selected channels:", [(i, channel_names[i] if i < len(channel_names) else str(i)) for i in selected_channels])


In [ ]:
std = torch.tensor(STD_LAYER, dtype=torch.float32, device=DEVICE).view(1, -1, 1, 1)
metrics_sum = {
    "norm_mae": 0.0,
    "norm_rmse": 0.0,
    "denorm_mae": 0.0,
}
count = 0
first_input = None
first_recon = None
first_error = None

started = time.time()
try:
    with torch.no_grad():
        for batch_idx, x in enumerate(loader):
            if batch_idx >= MAX_BATCHES:
                break
            x = x.to(DEVICE, non_blocking=True)
            recon, _ = model(x)
            err = recon - x
            batch = x.size(0)
            metrics_sum["norm_mae"] += err.abs().mean().item() * batch
            metrics_sum["norm_rmse"] += torch.sqrt((err ** 2).mean()).item() * batch
            metrics_sum["denorm_mae"] += ((recon * std - x * std).abs().mean()).item() * batch
            count += batch

            if first_input is None:
                first_input = x[0].detach().cpu()
                first_recon = recon[0].detach().cpu()
                first_error = err[0].abs().detach().cpu()
finally:
    if hasattr(dataset, "close"):
        dataset.close()

if count == 0:
    raise RuntimeError("No samples were evaluated. Check split and dataset config.")

metrics = {k: v / count for k, v in metrics_sum.items()}
metrics.update({
    "samples": count,
    "batches": min(MAX_BATCHES, len(loader)),
    "checkpoint": CKPT_PATH,
    "cfg": CFG_PATH,
    "split": SPLIT,
    "elapsed_sec": time.time() - started,
})
print(json.dumps(metrics, indent=2))
metrics_path = EVAL_OUTDIR / "ae_case_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"Saved metrics to {metrics_path}")


In [ ]:
def robust_vlim(arr):
    arr = np.asarray(arr)
    lo, hi = np.percentile(arr, [2, 98])
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        lo, hi = float(np.nanmin(arr)), float(np.nanmax(arr))
    return lo, hi

rows = len(selected_channels)
fig, axes = plt.subplots(rows, 3, figsize=(10, max(2.2 * rows, 3)), constrained_layout=True)
if rows == 1:
    axes = np.asarray([axes])
for r, ch in enumerate(selected_channels):
    inp = first_input[ch].numpy()
    rec = first_recon[ch].numpy()
    err = first_error[ch].numpy()
    name = channel_names[ch] if ch < len(channel_names) else f"ch{ch}"
    vmin, vmax = robust_vlim(np.concatenate([inp.reshape(-1), rec.reshape(-1)]))
    for c, (title, arr, cmap) in enumerate([
        (f"input {name}", inp, "viridis"),
        (f"recon {name}", rec, "viridis"),
        (f"abs err {name}", err, "magma"),
    ]):
        ax = axes[r, c]
        if c < 2:
            im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
        else:
            im = ax.imshow(arr, cmap=cmap)
        ax.set_title(title, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

fig_path = EVAL_OUTDIR / "ae_case_reconstruction.png"
fig.savefig(fig_path, dpi=140)
plt.close(fig)
print(f"Saved figure to {fig_path}")


In [ ]:
print("Evaluation complete.")
print("Artifacts:")
for path in sorted(EVAL_OUTDIR.glob("ae_case_*")):
    print(f"- {path} ({path.stat().st_size} bytes)")
